# Chapter 4

In [2]:
import torch
import tiktoken

from gpt_model import GPTModel

In [ ]:
# Does not work anymore, changes to GPTModel class have to be reverted
GPT_CONFIG_124M = {
    "vocab_size": 50257,
    "context_length": 1024,
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.1,
    "qkv_bias": False
}

tokenizer = tiktoken.get_encoding("gpt2")
batch = []
txt1 = "Every effort moves you"
txt2 = "Every day holds a"

batch.append(torch.tensor(tokenizer.encode(txt1)))
batch.append(torch.tensor(tokenizer.encode(txt2)))
batch = torch.stack(batch, dim=0)

torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)
out = model(batch)
print(out.shape)
out

KeyError: 'drop_rate_emb'

In [5]:
total_params = sum(p.numel() for p in model.parameters())
total_params

163009536

In [ ]:
# Calculates number of parameters in entire model
trf_blocks = [trf for trf in model.trf_blocks]
ffs = [trf.ff for trf in trf_blocks]
mhas = [trf.att for trf in trf_blocks]

ffs_params = sum(p.numel() for ff in ffs for p in ff.parameters())
mhas_params = sum(p.numel() for mha in mhas for p in mha.parameters())
print(ffs_params)
print(mhas_params)

56669184
28320768


In [14]:
def generate_text_simple(model, idx, max_new_tokens, context_size):
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]
        with torch.no_grad():
            logits = model(idx_cond)

        logits = logits[:, -1, :]
        probas = torch.softmax(logits, dim=-1)
        idx_next = torch.argmax(probas, dim=-1, keepdim=True)
        idx = torch.cat((idx, idx_next), dim=1)

    return idx

In [16]:
start_context = "Hello, I am"
encoded = tokenizer.encode(start_context)
print(f"{encoded=}")
encoded_tensor = torch.tensor(encoded).unsqueeze(0)
encoded_tensor.shape

encoded=[15496, 11, 314, 716]


torch.Size([1, 4])

In [17]:
model.eval()
out = generate_text_simple(
    model=model,
    idx=encoded_tensor,
    max_new_tokens=6,
    context_size=GPT_CONFIG_124M["context_length"]
)
print(out)

tensor([[15496,    11,   314,   716, 27018, 24086, 47843, 30961, 42348,  7267]])


In [18]:
decoded_text = tokenizer.decode(out.squeeze(0).tolist())
decoded_text

'Hello, I am Featureiman Byeswickattribute argue'

In [8]:
GPT_CONFIG_124M_MODIFIED = {
    "vocab_size": 50257,
    "context_length": 1024,
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate_attn": 0.1,
    "drop_rate_shortcut": 0.1,
    "drop_rate_emb": 0.1,
    "qkv_bias": False
}